# SpaceX Falcon 9 - Web Scraping Launch Records from Wikipedia

Scrape the *List of Falcon 9 and Falcon Heavy launches* page with BeautifulSoup and build a DataFrame.

In [ ]:
import requests
import re
import unicodedata
import pandas as pd
from bs4 import BeautifulSoup

: 

In [ ]:
static_url = 'https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922'
response = requests.get(static_url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
soup.title

In [ ]:
html_tables = soup.find_all('table')
print(len(html_tables))
first_launch_table = html_tables[2]
print(first_launch_table.text[:300])

In [ ]:
column_names = []
for th in first_launch_table.find_all('th'):
    name = ''.join(th.stripped_strings) if hasattr(th, 'stripped_strings') else ''
    name = re.sub(r'\[.*?\]', '', name).strip()
    if name and len(name) > 0:
        column_names.append(name)
print(column_names)

## Helper functions

In [ ]:
def date_time(table_cells):
    return [d.strip() for d in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return ''.join([b for i, b in enumerate(table_cells.strings) if i % 2 == 0][0:-1])

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize('NFKD', table_cells.text).strip()
    if mass:
        return mass[0:mass.find('kg') + 2]
    return 0

def safe_text(cell):
    if cell.a is not None and cell.a.string:
        return cell.a.string
    return cell.get_text(strip=True)

In [ ]:
launch_dict = dict.fromkeys(['Flight No.', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer',
                             'Launch outcome', 'Version Booster', 'Booster landing', 'Date', 'Time'])
for key in launch_dict:
    launch_dict[key] = []

extracted_row = 0
for table in soup.find_all('table', 'wikitable plainrowheaders collapsible'):
    for rows in table.find_all('tr'):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False
        row = rows.find_all('td')
        if flag and len(row) >= 9:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            dt = date_time(row[0])
            launch_dict['Date'].append(dt[0].strip(','))
            launch_dict['Time'].append(dt[1] if len(dt) > 1 else '')
            bv = booster_version(row[1])
            if not bv:
                bv = safe_text(row[1])
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(safe_text(row[2]))
            launch_dict['Payload'].append(safe_text(row[3]))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(safe_text(row[5]))
            launch_dict['Customer'].append(safe_text(row[6]))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))
print('Rows extracted:', extracted_row)

In [ ]:
df = pd.DataFrame({k: pd.Series(v) for k, v in launch_dict.items()})
df.head()

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)
print('Saved spacex_web_scraped.csv', df.shape)